# Entrenamiento YOLOv8 Multiclase en Google Colab (GPU NVIDIA T4)
### Sistema de Apoyo al Diagnóstico Odontológico para Detección de Caries en Fotografía Intraoral
**Proyecto e-Health | CRISP-DM Fase 4 (Modeling) & Fase 5 (Evaluation)**

Este notebook está diseñado para ejecutarse directamente en **Google Colab con entorno GPU T4** (16 GB VRAM), permitiendo entrenar el modelo baseline multiclase a máxima velocidad (~15-20 minutos para 50 épocas) y sin desgaste térmico en máquinas locales.

--- 
### Flujo de Trabajo Autónomo:
1. **Conexión e Instalación:** Montaje seguro de Google Drive e instalación silenciosa de `ultralytics`.
2. **Descompresión en Almacenamiento Local Efímero (`/content/dental-data/`):** Evita la severa latencia de I/O de Google Drive FUSE.
3. **Configuración Dinámica (`data_colab.yaml`):** Rutas absolutas locales, 2 clases (`0: caries_permanente`, `1: caries_decidua`) y controles sanos.
4. **Entrenamiento Acelerado con GPU T4:** `device=0`, `batch=16`, `imgsz=640`, `epochs=50`, FP16 habilitado (`half=True`), determinista con semilla `42`.
5. **Evaluación Formal en Test Set:** Extracción de $mAP_{50}$, $mAP_{50-95}$, Precision, Recall y cálculo del $F1\text{-Score}$ frente al umbral clínico de negocio ($F1 \ge 0.85$).
6. **Exportación de Artefactos:** Respaldo automático de `best.pt` y archivo comprimido de métricas a Google Drive, con descarga local directa para el backend.

## Celda 1: Montaje de Google Drive e Instalación de Dependencias

In [ ]:
# 1. Montaje de Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Instalación silenciosa de Ultralytics YOLO
!pip install -q ultralytics pyyaml

# 3. Verificación de GPU NVIDIA CUDA
import torch
assert torch.cuda.is_available(), (
    "¡ALERTA! GPU no detectada. Por favor vaya a: "
    "Entorno de ejecución -> Cambiar tipo de entorno de ejecución -> Seleccionar T4 GPU."
)
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"✓ Acelerador activo: {gpu_name} ({vram_gb:.1f} GB VRAM DISPONIBLE)")

## Celda 2: Descompresión en Almacenamiento Local Efímero (`/content/dental-data/`)

> **Principio de Rendimiento I/O:** Entrenar un modelo de Deep Learning leyendo miles de imágenes directamente desde Google Drive a través del sistema de archivos FUSE (`/content/drive/MyDrive/...`) degrada críticamente la velocidad por latencia de red. Descomprimir el dataset directamente en el disco SSD efímero local (`/content/dental-data/`) acelera el throughput de lectura más de un 500%.

In [ ]:
import os
import shutil
from pathlib import Path

# Rutas candidatas para localizar colab_caries_dataset.zip
POSSIBLE_ZIP_LOCATIONS = [
    Path("/content/colab_caries_dataset.zip"),
    Path("/content/drive/MyDrive/colab_caries_dataset.zip"),
    Path("/content/drive/MyDrive/dental-caries/colab_caries_dataset.zip"),
]

zip_file = next((p for p in POSSIBLE_ZIP_LOCATIONS if p.exists()), None)

if zip_file is None:
    print("[Aviso] No se encontró 'colab_caries_dataset.zip' en las rutas automáticas.")
    print("Por favor sube el archivo 'colab_caries_dataset.zip' mediante el siguiente selector:")
    from google.colab import files
    uploaded = files.upload()
    zip_file = Path("/content/colab_caries_dataset.zip")

DEST_DATA_DIR = Path("/content/dental-data")
if DEST_DATA_DIR.exists():
    shutil.rmtree(DEST_DATA_DIR)
DEST_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Descomprimiendo '{zip_file}' en {DEST_DATA_DIR}...")
!unzip -q "{zip_file}" -d "{DEST_DATA_DIR}"

# Verificación del contenido descomprimido
train_imgs = len(list((DEST_DATA_DIR / "dataset/selected/images/train").glob("*.jpg")))
val_imgs   = len(list((DEST_DATA_DIR / "dataset/selected/images/val").glob("*.jpg")))
test_imgs  = len(list((DEST_DATA_DIR / "dataset/selected/images/test").glob("*.jpg")))
total_imgs = train_imgs + val_imgs + test_imgs

print("=" * 60)
print(f"✓ Dataset listo en disco local efímero: {total_imgs} imágenes totales")
print(f"  - Train: {train_imgs} imágenes")
print(f"  - Val:   {val_imgs} imágenes")
print(f"  - Test:  {test_imgs} imágenes")
print("=" * 60)

## Celda 3: Generación Dinámica de `data_colab.yaml`
Generamos el archivo de configuración YOLO apuntando con rutas absolutas al almacenamiento local de Colab, con el esquema multiclase de 2 patologías.

In [ ]:
import yaml

DATASET_ROOT_COLAB = DEST_DATA_DIR / "dataset" / "selected"

colab_yaml_content = {
    "path": str(DATASET_ROOT_COLAB),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": 2,
    "names": ["caries_permanente", "caries_decidua"],
}

COLAB_YAML_PATH = DEST_DATA_DIR / "data_colab.yaml"
with open(COLAB_YAML_PATH, "w", encoding="utf-8") as f:
    yaml.dump(colab_yaml_content, f, sort_keys=False)

print(f"✓ Archivo de configuración generado: {COLAB_YAML_PATH}")
print("-" * 50)
print(open(COLAB_YAML_PATH).read())
print("-" * 50)

## Celda 4: Entrenamiento Controlado y Determinista en GPU T4
Entrenamos el detector utilizando:
- **Modelo:** `yolov8n.pt` (Nano baseline) o `yolov8s.pt` (Small para mayor capacidad).
- **Épocas:** 50 épocas con tamaño de lote `batch=16` (óptimo para los 16 GB de VRAM de la T4).
- **Precisión:** `half=True` (aceleración FP16 mediante Tensor Cores de la GPU T4).
- **Reproducibilidad:** `seed=42` y `deterministic=True`.

In [ ]:
from ultralytics import YOLO

# Configuración del Experimento en Colab
MODEL_NAME = "yolov8n.pt"   # Alternativa para mayor recall: 'yolov8s.pt'
EPOCHS = 50
BATCH_SIZE = 16
IMG_SIZE = 640
GLOBAL_SEED = 42
RUNS_DIR = Path("/content/runs")
EXP_NAME = "caries_yolo_colab"

# Instanciar modelo preentrenado para transfer learning
model = YOLO(MODEL_NAME)

print(f"Iniciando entrenamiento en {gpu_name} (50 épocas, batch={BATCH_SIZE}, half=True)...")

train_results = model.train(
    data=str(COLAB_YAML_PATH),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=0,
    seed=GLOBAL_SEED,
    deterministic=True,
    half=True,               # Aceleración nativa FP16 en GPU NVIDIA T4
    workers=4,
    project=str(RUNS_DIR),
    name=EXP_NAME,
    exist_ok=True,
    plots=True,
    save=True,
    save_period=10,
    verbose=True,
)

print("\n✓ Entrenamiento completado exitosamente.")

## Celda 5: Evaluación Formal en Test Set y Validación de Regla de Negocio ($F1 \ge 0.85$)
Evaluamos los mejores pesos (`best.pt`) contra el conjunto de test independiente (305 imágenes de 153 pacientes reservados). Calculamos las métricas globales y desglosadas por clase para validar formalmente la regla de negocio clínica.

In [ ]:
BEST_WEIGHTS_PATH = RUNS_DIR / EXP_NAME / "weights" / "best.pt"
assert BEST_WEIGHTS_PATH.exists(), f"ERROR: No se encontró {BEST_WEIGHTS_PATH}"

# Cargar mejores pesos para inferencia y evaluación
eval_model = YOLO(str(BEST_WEIGHTS_PATH))

print("Evaluando modelo en el conjunto de prueba (Test Split)...")
test_metrics = eval_model.val(
    data=str(COLAB_YAML_PATH),
    split="test",
    project=str(RUNS_DIR),
    name=f"{EXP_NAME}_test_eval",
    plots=True,
)

# Extracción de métricas clave
precision = float(test_metrics.box.mp)
recall    = float(test_metrics.box.mr)
map50     = float(test_metrics.box.map50)
map50_95  = float(test_metrics.box.map)

# Cálculo matemático formal del F1-Score: 2 * (P * R) / (P + R)
if (precision + recall) > 0:
    f1_score = 2.0 * (precision * recall) / (precision + recall)
else:
    f1_score = 0.0

print("\n" + "=" * 75)
print(" REPORTE DE EVALUACIÓN CLÍNICA EN CONJUNTO DE PRUEBA (TEST SET)")
print("=" * 75)
print(f"  Precision (P):       {precision:.4f} ({precision*100:.2f}%)")
print(f"  Recall (R):          {recall:.4f} ({recall*100:.2f}%)")
print(f"  mAP@50:              {map50:.4f} ({map50*100:.2f}%)")
print(f"  mAP@50-95:           {map50_95:.4f} ({map50_95*100:.2f}%)")
print(f"  F1-Score Combinado:  {f1_score:.4f} ({f1_score*100:.2f}%)")
print("-" * 75)

# Desglose por Patología Odontológica (si está disponible en el objeto box)
try:
    class_maps = test_metrics.box.maps
    for cls_idx, cls_name in enumerate(["caries_permanente", "caries_decidua"]):
        if cls_idx < len(class_maps):
            print(f"  - {cls_name:18}: mAP@50-95 = {class_maps[cls_idx]:.4f}")
except Exception:
    pass

print("-" * 75)
# Validación de Regla de Negocio del Sistema e-Health
BUSINESS_THRESHOLD_F1 = 0.85
if f1_score >= BUSINESS_THRESHOLD_F1:
    print(f"✓ REGLA DE NEGOCIO CUMPLIDA: F1-Score ({f1_score:.4f}) >= {BUSINESS_THRESHOLD_F1}")
    print("  El modelo califica para pase a producción en el backend clínico.")
else:
    print(f"⚠ REGLA DE NEGOCIO EN OBSERVACIÓN: F1-Score ({f1_score:.4f}) < {BUSINESS_THRESHOLD_F1}")
    print("  Recomendación: Entrenar 'yolov8s.pt' o extender a 100 épocas con aumento de datos.")
print("=" * 75)

## Celda 6: Exportación Automatizada de Artefactos a Google Drive y Descarga Local

In [ ]:
import shutil
from google.colab import files

# Directorio de respaldo seguro en Google Drive
DRIVE_ARTIFACTS_DIR = Path("/content/drive/MyDrive/dental_caries_artifacts")
DRIVE_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

# 1. Guardar pesos best.pt en Google Drive
drive_model_target = DRIVE_ARTIFACTS_DIR / "best.pt"
shutil.copy2(BEST_WEIGHTS_PATH, drive_model_target)
print(f"✓ Pesos entrenados respaldados en Google Drive: {drive_model_target}")

# 2. Empaquetar curvas de aprendizaje, matrices de confusión y logs
local_zip_runs = "/content/caries_colab_training_runs.zip"
shutil.make_archive("/content/caries_colab_training_runs", "zip", str(RUNS_DIR))
drive_runs_target = DRIVE_ARTIFACTS_DIR / "caries_colab_training_runs.zip"
shutil.copy2(local_zip_runs, drive_runs_target)
print(f"✓ Gráficos y curvas respaldados en Google Drive: {drive_runs_target}")

# 3. Disparar descarga directa del archivo best.pt al navegador local
print("\nIniciando descarga directa de 'best.pt' para transferir al backend...")
try:
    files.download(str(BEST_WEIGHTS_PATH))
    print("✓ Descarga iniciada en su navegador.")
except Exception as e:
    print(f"[Nota] Puede descargar 'best.pt' directamente desde su Google Drive: {drive_model_target}")